<a href="https://colab.research.google.com/github/TanviDeore/ASL_Recognition_Grp14/blob/main/ASL_OpticalFlow_Features_v2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ASL Recognition — Optical Flow Features v2 (Spatial Grid + Temporal Chunks)

**Standalone notebook — run from top to bottom.** Produces `artifacts_v2/` with a 132-dim per-video feature.

**What's different from `ASL_OpticalFlow_Features.ipynb` (v1):**

v1 averaged HOOF over the whole frame and over the whole video. That destroyed two things the classifier needs:

- **Where** motion happens (head vs chest vs left/right hand)
- **When** motion happens (up→down vs down→up)

v2 fixes both with four changes that stay inside the optical-flow scope (no MediaPipe, no deep learning):

1. **Spatial pooling — 2×2 grid.** HOOF is computed per quadrant, not per whole frame.
2. **Temporal pooling — 3 chunks.** HOOF is averaged within beginning/middle/end thirds, then concatenated.
3. **Background suppression.** Pixels with magnitude below 0.2 px don't vote in the HOOF — kills static-background noise.
4. **Power + L2 normalization.** Standard signed-sqrt + unit-norm trick from action recognition.

**v2 dimension** = 3 chunks × (4 cells × (9 HOOF bins + 1 cell mean magnitude) + 4 global stats) = **132 dims per video**.

**Output folder is `artifacts_v2/`** — kept separate from the v1 `artifacts/` folder so both coexist in the repo for the report's ablation comparison.

## 0. Setup — install deps, fetch metadata, download frames

In [1]:
!pip install -q opencv-python numpy matplotlib tqdm requests mediapipe


In [2]:
# Pull metadata file from the repo (it lives under data/ in the repo)
import os, urllib.request

url = 'https://raw.githubusercontent.com/TanviDeore/ASL_Recognition_Grp14/main/data/nslt_100.json'
urllib.request.urlretrieve(url, 'nslt_100.json')

assert os.path.exists('nslt_100.json') and os.path.getsize('nslt_100.json') > 0, 'Download failed'
print(f'Downloaded nslt_100.json — {os.path.getsize("nslt_100.json")/1024:.1f} KB')

Downloaded nslt_100.json — 104.6 KB


In [9]:
# Download preprocessed frames from Box. Skips if frames are already present (re-run safe).
import os, requests, zipfile
from tqdm import tqdm

def download_data():
    url = 'https://utdallas.box.com/shared/static/5ry2jpk80fxvdq9xet51q65o3sd6bx0v.zip'
    output_zip = 'asl_frames.zip'
    extract_to = 'data/frames'

    if os.path.exists(extract_to) and len(os.listdir(extract_to)) > 0:
        print(f'Frames already present at {extract_to} ({len(os.listdir(extract_to))} folders) — skipping download.')
        return

    os.makedirs('data', exist_ok=True)
    print('Downloading dataset from Box...')
    response = requests.get(url, stream=True)
    response.raise_for_status()
    total_size = int(response.headers.get('content-length', 0))
    with open(output_zip, 'wb') as f, tqdm(
        total=total_size, unit='iB', unit_scale=True, unit_divisor=1024
    ) as bar:
        for chunk in response.iter_content(chunk_size=8192):
            bar.update(f.write(chunk))

    print('\nExtracting frames...')
    with zipfile.ZipFile(output_zip, 'r') as zip_ref:
        zip_ref.extractall(extract_to)
    os.remove(output_zip)
    print(f'Done. {len(os.listdir(extract_to))} folders ready in {extract_to}')

download_data()

100%|██████████| 1.57G/1.57G [00:27<00:00, 62.0MiB/s]



Extracting frames...
Done. 1013 folders ready in data/frames


## 1. Build the work list

`nslt_100.json` has 2038 entries; only ~1013 folders were extracted (the rest are listed in `missing.txt`). Process only what we have.

In [18]:
import json
from collections import Counter

with open('nslt_100.json') as f:
    meta = json.load(f)

frames_root = 'data/frames'
available_video_ids = set(os.listdir(frames_root))

work_list = []
for vid, info in meta.items():
    if vid not in available_video_ids:
        continue
    work_list.append({
        'video_id': vid,
        'class_idx': info['action'][0],
        'subset': info.get('subset', 'unknown'),
    })

print(f'Videos to process: {len(work_list)}')
print(f'Subset distribution: {dict(Counter(w["subset"] for w in work_list))}')
print(f'Number of unique classes: {len(set(w["class_idx"] for w in work_list))}')

class_counts = Counter(w["class_idx"] for w in work_list)

print("Top classes by frequency:")
print(class_counts.most_common(10))

TOP_K = 30

top_classes = [cls for cls, _ in class_counts.most_common(TOP_K)]

print(f"Selected top {TOP_K} classes:")
print(top_classes)

filtered_work_list = [w for w in work_list if w["class_idx"] in top_classes]

print(f"Videos after filtering: {len(filtered_work_list)}")
print(f"New class count: {len(set(w['class_idx'] for w in filtered_work_list))}")

Videos to process: 1013
Subset distribution: {'val': 165, 'train': 748, 'test': 100}
Number of unique classes: 100
Top classes by frequency:
[(3, 16), (20, 16), (14, 16), (1, 15), (5, 15), (7, 14), (12, 14), (9, 14), (2, 14), (48, 13)]
Selected top 30 classes:
[3, 20, 14, 1, 5, 7, 12, 9, 2, 48, 32, 29, 8, 72, 51, 34, 46, 45, 17, 77, 59, 30, 65, 68, 56, 54, 44, 69, 36, 52]
Videos after filtering: 390
New class count: 30


## 2. Farnebäck dense optical flow

Same flow computation as v1 — frames resized to 160×120, Farnebäck dense flow between consecutive grayscale frames. Only the post-processing changes in v2.

In [19]:
import cv2
import numpy as np
import mediapipe as mp
import os

FLOW_SIZE = (160, 120)  # (width, height)

mp_holistic = mp.solutions.holistic

def compute_flow(prev_gray, next_gray):
    return cv2.calcOpticalFlowFarneback(
        prev_gray, next_gray, None,
        pyr_scale=0.5, levels=3, winsize=15,
        iterations=3, poly_n=5, poly_sigma=1.2, flags=0,
    )

def load_video_frames(video_dir):
    files = sorted(f for f in os.listdir(video_dir) if f.endswith('.jpg'))
    frames = []
    for fn in files:
        img = cv2.imread(os.path.join(video_dir, fn))
        if img is not None:
            frames.append(img)
    return frames

def get_bounding_box_and_landmarks(frame, holistic):
    # Convert the BGR image to RGB before processing.
    results = holistic.process(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    
    H, W = frame.shape[:2]
    xs, ys = [], []
    
    # 21 points each for hands
    left_hand_lms = np.zeros(21 * 2, dtype=np.float32)
    right_hand_lms = np.zeros(21 * 2, dtype=np.float32)
    
    if results.pose_landmarks:
        for lm in results.pose_landmarks.landmark:
            xs.append(lm.x * W)
            ys.append(lm.y * H)
            
    if results.left_hand_landmarks:
        for i, lm in enumerate(results.left_hand_landmarks.landmark):
            xs.append(lm.x * W)
            ys.append(lm.y * H)
            left_hand_lms[i*2] = lm.x
            left_hand_lms[i*2+1] = lm.y
            
    if results.right_hand_landmarks:
        for i, lm in enumerate(results.right_hand_landmarks.landmark):
            xs.append(lm.x * W)
            ys.append(lm.y * H)
            right_hand_lms[i*2] = lm.x
            right_hand_lms[i*2+1] = lm.y
            
    if len(xs) > 0:
        min_x, max_x = max(0, int(min(xs)) - 20), min(W, int(max(xs)) + 20)
        min_y, max_y = max(0, int(min(ys)) - 20), min(H, int(max(ys)) + 20)
        bbox = (min_x, min_y, max_x, max_y)
    else:
        bbox = (0, 0, W, H)
        
    landmarks = np.concatenate([left_hand_lms, right_hand_lms])
    return bbox, landmarks

def video_flows_and_landmarks(video_dir):
    frames = load_video_frames(video_dir)
    if len(frames) < 2:
        return [], []
        
    flows = []
    landmarks_list = []
    
    with mp_holistic.Holistic(static_image_mode=False, min_detection_confidence=0.5) as holistic:
        prev_frame = cv2.resize(frames[0], FLOW_SIZE)
        prev_gray = cv2.cvtColor(prev_frame, cv2.COLOR_BGR2GRAY)
        
        for i in range(1, len(frames)):
            curr_frame = cv2.resize(frames[i], FLOW_SIZE)
            next_gray = cv2.cvtColor(curr_frame, cv2.COLOR_BGR2GRAY)
            
            bbox, landmarks = get_bounding_box_and_landmarks(curr_frame, holistic)
            landmarks_list.append(landmarks)
            
            flow = compute_flow(prev_gray, next_gray)
            
            # Mask out background outside bbox
            mask = np.zeros_like(flow[..., 0])
            min_x, min_y, max_x, max_y = bbox
            mask[min_y:max_y, min_x:max_x] = 1.0
            
            flow[..., 0] *= mask
            flow[..., 1] *= mask
            
            flows.append(flow)
            prev_gray = next_gray
            
    return flows, landmarks_list

# Sanity check
sample_dir = os.path.join(frames_root, filtered_work_list[0]['video_id'])
flows, lms = video_flows_and_landmarks(sample_dir)
print(f'Video {filtered_work_list[0]["video_id"]}: {len(flows)} flow pairs, each of shape {flows[0].shape}')


I0000 00:00:1777521996.864891 2710384 gl_context.cc:357] GL version: 2.1 (2.1 Metal - 89.3), renderer: Apple M3
W0000 00:00:1777521996.960451 2789419 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1777521996.972510 2789417 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1777521996.975930 2789415 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1777521996.976861 2789418 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1777521996.976878 2789420 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support 

Video 51068: 24 flow pairs, each of shape (120, 160, 2)


## 3. v2 feature extractor (the new part)

Per flow field → 44-dim vector:
- **4 spatial cells × (9 HOOF bins + 1 cell mean magnitude)** = 40 dims (with mag threshold + power+L2 norm per cell)
- **4 global stats** (mean_mag, std_mag, p90_mag, mean_ang) = 4 dims

Per video → 132-dim vector: stack 3 temporal chunks (mean within each).

In [26]:
V2_GRID = (2, 2)          # rows × cols = 4 spatial cells
V2_TEMPORAL_CHUNKS = 5    # Increased to 5 for better temporal resolution
V2_ANGLE_BINS = 9
V2_MAG_THRESHOLD = 0.2    # pixels with mag below this don't vote in HOOF
V2_PER_PAIR_FLOW_DIM = V2_GRID[0] * V2_GRID[1] * (V2_ANGLE_BINS + 1) + 4   # 44
V2_PER_PAIR_LM_DIM = 21 * 2 * 2  # 84 (left hand + right hand x,y)
V2_PER_PAIR_DIM = V2_PER_PAIR_FLOW_DIM + V2_PER_PAIR_LM_DIM # 128
V2_FEATURE_DIM = V2_TEMPORAL_CHUNKS * V2_PER_PAIR_DIM * 2          # 1280

def _power_l2(v, alpha=0.5, eps=1e-9):
    """Signed power norm + L2 — standard for histogram features feeding an SVM."""
    v = np.sign(v) * np.abs(v) ** alpha
    return v / (np.linalg.norm(v) + eps)

def per_pair_features_v2(flow, landmarks):
    """Convert one flow field and landmarks into a 128-dim feature vector."""
    fx, fy = flow[..., 0], flow[..., 1]
    mag, ang = cv2.cartToPolar(fx, fy)
    H, W = mag.shape
    rows, cols = V2_GRID
    rh, cw = H // rows, W // cols

    cell_features = []
    for r in range(rows):
        for c in range(cols):
            cell_m = mag[r*rh:(r+1)*rh, c*cw:(c+1)*cw].ravel()
            cell_a = ang[r*rh:(r+1)*rh, c*cw:(c+1)*cw].ravel()
            mask = cell_m > V2_MAG_THRESHOLD

            if mask.sum() < 10:
                cell_hoof = np.zeros(V2_ANGLE_BINS, dtype=np.float32)
            else:
                hoof, _ = np.histogram(
                    cell_a[mask], bins=V2_ANGLE_BINS,
                    range=(0, 2 * np.pi), weights=cell_m[mask],
                )
                cell_hoof = _power_l2(hoof.astype(np.float32))

            cell_mean_mag = float(cell_m.mean())
            cell_features.append(np.concatenate([cell_hoof, [cell_mean_mag]]))

    spatial = np.concatenate(cell_features)  # 4 × 10 = 40

    p90 = float(np.percentile(mag, 90))
    global_stats = np.array(
        [float(mag.mean()), float(mag.std()), p90, float(ang.mean())],
        dtype=np.float32,
    )

    flow_features = np.concatenate([spatial, global_stats]).astype(np.float32)  # 44
    return np.concatenate([flow_features, landmarks]).astype(np.float32) # 128

def video_features_chunks(video_dir, num_chunks=5):
    flows, landmarks_seq = video_flows_and_landmarks(video_dir)
    if len(flows) < num_chunks:
        return []  # skip very short videos

    per_pair = np.stack([per_pair_features_v2(f, lm) for f, lm in zip(flows, landmarks_seq)])  # (N_pairs, D)

    chunk_size = len(per_pair) // num_chunks
    features = []

    for i in range(num_chunks):
        start = i * chunk_size
        end = (i + 1) * chunk_size if i < num_chunks - 1 else len(per_pair)

        chunk = per_pair[start:end]

            # aggregate within chunk
        feat = np.concatenate([
            chunk.mean(axis=0),
            chunk.std(axis=0)
        ])

        features.append(feat.astype(np.float32))

    return np.concatenate(features)

# Sanity check on the first video
feat = video_features_chunks(os.path.join(frames_root, filtered_work_list[0]['video_id']))
print(f'v2 feature dim: {len(feat)} (expected {V2_FEATURE_DIM})')


I0000 00:00:1777522375.918161 2710384 gl_context.cc:357] GL version: 2.1 (2.1 Metal - 89.3), renderer: Apple M3
W0000 00:00:1777522376.003985 2796250 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1777522376.017875 2796248 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1777522376.021825 2796248 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1777522376.022043 2796250 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1777522376.022870 2796249 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support 

v2 feature dim: 1280 (expected 1280)


## 4. Full extraction → save to `artifacts_v2/`

~6 minutes on Colab CPU. Writes everything to a brand-new `artifacts_v2/` folder; does not touch the v1 `artifacts/` folder.

In [27]:
def extract_all_v2(work_subset, desc='Extracting v2'):
    X, y, ids, subs = [], [], [], []
    skipped = 0
    
    for w in tqdm(work_subset, desc=desc):
        feat_seq = video_features_v2(os.path.join(frames_root, w['video_id']))
        if feat_seq is None:
            skipped += 1
            continue

        feat = np.mean(feat_seq, axis=0)

        X.append(feat)
        y.append(w['class_idx'])
        ids.append(w['video_id'])
        subs.append(w['subset'])

    return np.stack(X).astype(np.float32), np.array(y, dtype=np.int64), ids, subs, skipped

X, y, ids, subs, skipped = extract_all_v2(filtered_work_list, desc='v2 full pass')

print(f'\nv2 features shape: {X.shape}')
print(f'v2 labels shape:   {y.shape}')
print(f'Skipped:           {skipped}')
print(f'Subset distribution: {dict(Counter(subs))}')
print(f'Class count:         {len(set(y.tolist()))}')

v2 full pass:   0%|          | 0/390 [00:00<?, ?it/s]I0000 00:00:1777522381.240410 2710384 gl_context.cc:357] GL version: 2.1 (2.1 Metal - 89.3), renderer: Apple M3
W0000 00:00:1777522381.387692 2796376 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1777522381.402023 2796373 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1777522381.403387 2796374 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1777522381.403628 2796375 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1777522381.404223 2796378 inference_feedback_manager.cc:114] Feedback manager requires a model 


v2 features shape: (390, 256)
v2 labels shape:   (390,)
Skipped:           0
Subset distribution: {'test': 44, 'train': 280, 'val': 66}
Class count:         30


In [28]:
# Save artifacts to a SEPARATE folder — does not touch the v1 artifacts/
os.makedirs('artifacts_v2', exist_ok=True)

np.save('artifacts_v2/features.npy', X)
np.save('artifacts_v2/labels.npy', y)

splits = {
    'train': [ids[i] for i, s in enumerate(subs) if s == 'train'],
    'val':   [ids[i] for i, s in enumerate(subs) if s == 'val'],
    'test':  [ids[i] for i, s in enumerate(subs) if s == 'test'],
}
with open('artifacts_v2/splits.json', 'w') as f:
    json.dump(splits, f, indent=2)
with open('artifacts_v2/video_ids.json', 'w') as f:
    json.dump(ids, f, indent=2)

with open('artifacts_v2/feature_schema.json', 'w') as f:
    json.dump({
        'feature_dim': int(X.shape[1]),
        'version': 'v2',
        'layout': '5 temporal chunks × (4 spatial cells × (9 HOOF bins + 1 cell mean_mag) + 4 global stats + 84 hand landmarks)',
        'spatial_grid_rows_cols': list(V2_GRID),
        'temporal_chunks': V2_TEMPORAL_CHUNKS,
        'angle_bins': V2_ANGLE_BINS,
        'mag_threshold_for_hoof': V2_MAG_THRESHOLD,
        'cell_normalization': 'signed sqrt (alpha=0.5) + L2 per cell HOOF',
        'flow_method': 'Farneback dense optical flow with MediaPipe background masking',
        'flow_resize_wh': list(FLOW_SIZE),
        'aggregation': 'mean within each temporal chunk, then concatenate chunks',
        'num_classes': int(len(set(y.tolist()))),
        'num_videos': int(X.shape[0]),
        'baseline_artifacts_dir': '../artifacts/  (v1 — 20-dim, mean across whole video) for ablation',
    }, f, indent=2)

print('Saved v2 artifacts (the v1 artifacts/ folder was NOT touched):')
for fn in sorted(os.listdir('artifacts_v2')):
    size = os.path.getsize(f'artifacts_v2/{fn}')
    print(f'  artifacts_v2/{fn}  ({size/1024:.1f} KB)')

Saved v2 artifacts (the v1 artifacts/ folder was NOT touched):
  artifacts_v2/feature_schema.json  (0.7 KB)
  artifacts_v2/features.npy  (390.1 KB)
  artifacts_v2/labels.npy  (3.2 KB)
  artifacts_v2/splits.json  (5.0 KB)
  artifacts_v2/video_ids.json  (4.2 KB)


In [9]:
# Bundle and download — only the new artifacts_v2/ folder.
import shutil
from google.colab import files

shutil.make_archive('artifacts_v2_bundle', 'zip', 'artifacts_v2')
files.download('artifacts_v2_bundle.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 5. Hand-off

After this notebook finishes, the repo will contain **two artifact folders**:

| Folder | Feature dim | Use |
|---|---|---|
| `artifacts/` | 20 (v1) | Baseline — for the report's ablation |
| `artifacts_v2/` | 132 (v2) | **Primary — train your classifier on this** |

Both folders have **identical filenames** (`features.npy`, `labels.npy`, `splits.json`, `video_ids.json`, `feature_schema.json`). Switching between them is a one-line path change.

### Starter snippet (Step 8) — try multiple classifiers

```python
import json, numpy as np
from sklearn.svm import LinearSVC, SVC
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier

ART = 'artifacts_v2'   # ← change to 'artifacts' to run the v1 baseline

X = np.load(f'{ART}/features.npy')
y = np.load(f'{ART}/labels.npy')
with open(f'{ART}/video_ids.json') as f: ids = json.load(f)
with open(f'{ART}/splits.json')    as f: splits = json.load(f)

id_to_row = {v: i for i, v in enumerate(ids)}
def rows(name): return [id_to_row[v] for v in splits[name] if v in id_to_row]
tr, va, te = rows('train'), rows('val'), rows('test')
trva = tr + va

scaler = StandardScaler().fit(X[trva])
Xs = scaler.transform(X)

print(f'[{ART}]  train+val={len(trva)}  test={len(te)}  feature_dim={X.shape[1]}\n')

for C in [0.1, 1.0, 10.0]:
    clf = LinearSVC(C=C, max_iter=10000, dual='auto').fit(Xs[trva], y[trva])
    print(f'LinearSVC  C={C:5}: test acc = {clf.score(Xs[te], y[te]):.3f}')

for C in [1.0, 10.0]:
    clf = SVC(kernel='rbf', C=C, gamma='scale').fit(Xs[trva], y[trva])
    print(f'SVC rbf    C={C:5}: test acc = {clf.score(Xs[te], y[te]):.3f}')

for k in [1, 3, 5]:
    clf = KNeighborsClassifier(n_neighbors=k).fit(Xs[trva], y[trva])
    print(f'kNN        k={k}    : test acc = {clf.score(Xs[te], y[te]):.3f}')
```

### Important notes

- **Always standardize before SVM.**
- **Don't shuffle the data randomly.** Use `splits.json` — the WLASL splits are signer-aware.